# Part 1. Equation of a Slime

#### Late Days
Zero

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

import warnings
warnings.filterwarnings('ignore')

## 1. Loading the dataset

In [2]:
science = pd.read_csv('science_data_large.csv')
science.head(15)

,Temperature °C,Mols KCL,Size nm^3
0,469,647,6.244743e+05
1,403,694,5.779610e+05
2,302,975,6.196847e+05
3,779,916,1.460449e+06
4,901,18,4.325726e+04
5,545,637,7.124634e+05
6,660,519,7.006960e+05
7,143,869,2.718260e+05
8,89,461,8.919803e+04
9,294,776,4.770210e+05


In [3]:
science.describe()

,Temperature °C,Mols KCL,Size nm^3
count,1000.000000,1000.000000,1.000000e+03
mean,500.500000,471.530000,5.086111e+05
std,288.819436,288.482872,4.474838e+05
min,1.000000,1.000000,1.611429e+01
25%,250.750000,226.750000,1.298267e+05
50%,500.500000,459.500000,3.827182e+05
75%,750.250000,710.250000,7.603211e+05
max,1000.000000,1000.000000,1.972127e+06


## 2. Splitting the dataset

In [4]:
X = science[['Temperature °C', 'Mols KCL']]
y = science['Size nm^3']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

## 3. Perform a Linear Regression

In [5]:
reg = LinearRegression().fit(X, y)
coef = reg.coef_
intercept = reg.intercept_

coef, intercept

(array([ 875.90992708, 1031.59502452]), -416209.8173862055)

After performing linear regression, we can define the equation for the Size of Slime as:

$$h(x) = -416209.8173 + 875.90993(T) + 1031.59502(M)$$

Where $h(x) = \text{Size } nm^3$, $T = \text{Temperature } °C$, and $M = \text{Mols } KCL$.

In [6]:
score = reg.score(X, y)
score

0.8607060208080753

The `score` refers to the $R^2$ (coefficient of determination) score. It measures the proportion of variance in the dependent variable that is predictable from the independent variables, telling us how good of a fit the linear model is for the data. 

An $R^2$ value of $0.86$ is in general considered pretty strong and means that the linear regression model is a good fit for the data. 

In [7]:
prediction = reg.predict(np.array([[500.0, 500.0]]))
prediction

array([537542.65841344])

A slime with a Temperature of $500.0 °C$ and a Molar Mass of $500.0 \text{ Mols } KCL$ would have a Size of approximately $537542.65841 \text{ }nm^3$

## 4. Use Cross Validation

In [8]:
cv_scores = cross_val_score(reg, X, y, cv=5, scoring='r2')
cv_scores, np.mean(cv_scores), np.std(cv_scores)

(array([0.83918826, 0.87051239, 0.85871066, 0.87202623, 0.84364641]),
 0.8568167899144437,
 0.01346630737209602)

When we repeat the experiment across multiple shuffles (using 5 splits) we see that the $R^2$ value stays somewhat the same, with the lowest score being 0.84 and the highest being 0.87. The mean is 0.86 and the standard deviation is 0.01 so there is very little variation. 

## 5. Using Polynomial Regression

In [9]:
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)
poly_model = LinearRegression()
poly_model.fit(X_poly, y)

intercept = poly_model.intercept_
coefficients = poly_model.coef_

intercept, coefficients

(1.6572477761656046e-05,
 array([ 0.00000000e+00,  1.20000000e+01, -1.23110930e-07, -1.05648729e-11,
         2.00000000e+00,  2.85714287e-02]))

After performing polynomial regression, we can define the equation for the Size of Slime as:

$$h(x) = 1.65724 \times 10^{-5} + 12.00000 X_1 - 1.23111 \times 10^{-7} X_2 - 1.05649 \times 10^{-11} X_3 + 2.00000 X_4 + 2.85714 \times 10^{-2}$$

Where $h(x) = \text{Size } nm^3$, $X_1 = \text{Temperature } °C$, $X_2 = \text{Mols } KCL$, $X_3 = \text{Temperature squared}$, $X_4 = \text{Interaction term}$, and $X_5 = \text{Mols KCL squared}$.

In [10]:
cv_scores = cross_val_score(poly_model, X_poly, y, cv=5, scoring='r2')
cv_scores, np.mean(cv_scores), np.std(cv_scores)

(array([1., 1., 1., 1., 1.]), 1.0, 0.0)

While these $R^2$ scores are unusually high and we have zero variation, the dataset used for this is synthetic, so a score as high as 1 is expected.

# Part 2. Chronic Kidney Disease Prediction via Classification

## 1. Loading the dataset

In [11]:
ckd = pd.read_csv('ckd_feature_subset.csv')
ckd.head(3)

,age,bp,wbcc,appet_poor,appet_good,rbcc,Target_ckd
0,0.688312,0.333333,0.000000,1,0,0.000000,1
1,0.545455,0.333333,0.128319,1,0,0.305085,1
2,0.714286,0.500000,0.238938,1,0,0.186441,1


In [12]:
ckd.describe()

,age,bp,wbcc,appet_poor,appet_good,rbcc,Target_ckd
count,153.000000,153.000000,153.000000,153.000000,153.000000,153.000000,153.000000
mean,0.563280,0.400871,0.206056,0.124183,0.875817,0.472361,0.274510
std,0.202485,0.183809,0.140267,0.330873,0.330873,0.174521,0.447733
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.428571,0.166667,0.119469,0.000000,1.000000,0.406780,0.000000
50%,0.571429,0.500000,0.172566,0.000000,1.000000,0.474576,0.000000
75%,0.701299,0.500000,0.261062,0.000000,1.000000,0.593220,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [13]:
ckd.columns.tolist()

['age', 'bp', 'wbcc', 'appet_poor', 'appet_good', 'rbcc', 'Target_ckd']

## 2. Pre-processing

In [14]:
scaler = StandardScaler()
ckd[['age', 'bp', 'wbcc', 'rbcc']] = scaler.fit_transform(ckd[['age', 'bp', 'wbcc', 'rbcc']])
ckd.head(3)

,age,bp,wbcc,appet_poor,appet_good,rbcc,Target_ckd
0,0.619514,-0.368643,-1.473855,1,0,-2.715508,1
1,-0.088322,-0.368643,-0.556031,1,0,-0.961636,1
2,0.748212,0.541073,0.235196,1,0,-1.643698,1


## 3. Splitting the dataset

In [15]:
X = ckd[['age', 'bp', 'wbcc', 'appet_poor', 'appet_good', 'rbcc']]
y = ckd['Target_ckd']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

## 4. Classification

## Results and Conclusion for Classification Experiments

### Logistic Regression Performance

In [16]:
log = LogisticRegression(random_state=42).fit(X, y)
cv_scores = cross_val_score(log, X, y, cv=5, scoring='accuracy')
cv_scores, np.mean(cv_scores), np.std(cv_scores)

(array([1.        , 0.93548387, 0.93548387, 0.93333333, 0.93333333]),
 0.9475268817204302,
 0.026254180579839755)

### SVC Performance

In [17]:
svc = SVC(gamma='auto', random_state=42).fit(X, y)
cv_scores = cross_val_score(svc, X, y, cv=5, scoring='accuracy')
cv_scores, np.mean(cv_scores), np.std(cv_scores)

(array([0.93548387, 0.96774194, 0.96774194, 1.        , 0.9       ]),
 0.9541935483870969,
 0.03391855336282559)

### K-Nearest-Neighbors Performance

In [18]:
knn = KNeighborsClassifier(n_neighbors=3).fit(X, y)
cv_scores = cross_val_score(knn, X, y, cv=5, scoring='accuracy')
cv_scores, np.mean(cv_scores), np.std(cv_scores)

(array([0.93548387, 1.        , 0.93548387, 1.        , 0.86666667]),
 0.9475268817204302,
 0.049669505490759734)

### Classification Results

| Model  | Average of Scores | Standard Deviation in Scores |
|--------|------------------|-----------------------------|
| log | 0.94753 | 0.02625 |
| svc | 0.95419 | 0.03392 |
| knn | 0.94752 | 0.04967 |

For classifiers, we look at accuracy instead of $R^2$. The results show us that using the SVC classifier gave us the highest accuracy score of 0.95419 with the Logistic classifier and KNN Classifier having accuracies of 0.94753 and 0.94752 respectively. The KNN Classifier had the most variation with a standard deviation of 0.04967 with SVC and Logistic classifiers having lower standard deviations on 0.03392 and 0.02625 respectively.

The Support Vector Classifier (SVC) performed the best, achieving the highest accuracy of 0.95419 with a relatively low standard deviation of 0.03392, meaning it was consistently accurate across different folds of cross-validation.

## 6. Neural Networks

### Default Settings

In [19]:
neural = MLPClassifier(random_state=42).fit(X_train, y_train)
cv_scores = cross_val_score(neural, X, y, cv=5, scoring='accuracy')
cv_scores, np.mean(cv_scores), np.std(cv_scores)

(array([1.        , 0.96774194, 0.90322581, 0.96666667, 0.93333333]),
 0.9541935483870969,
 0.03307635944899851)

The default settings for the Neural Network MLPClassifier give us an accuracy of 0.95 and a standard deviation of 0.03. We can attempt to improve these results by tinkering with some of the model parameters. 

### Hidden Layer Sizes

In [20]:
# deeper network
neural = MLPClassifier(hidden_layer_sizes=(32, 16, 8), random_state=42).fit(X_train, y_train)
cv_scores = cross_val_score(neural, X, y, cv=5, scoring='accuracy')
cv_scores, np.mean(cv_scores), np.std(cv_scores)

(array([1.        , 0.90322581, 0.96774194, 0.96666667, 0.96666667]),
 0.9608602150537635,
 0.031522803356622414)

In [21]:
# wider network
neural = MLPClassifier(hidden_layer_sizes=(64, 32), random_state=42).fit(X_train, y_train)
cv_scores = cross_val_score(neural, X, y, cv=5, scoring='accuracy')
cv_scores, np.mean(cv_scores), np.std(cv_scores)

(array([0.96774194, 0.96774194, 0.96774194, 1.        , 0.93333333]),
 0.9673118279569893,
 0.02108843126388174)

Hidden Layer Sizes defines the number of layers and neurons per layer. We can see that a deeper network like `(32, 16, 8)` had an accuracy of 0.960860 and a standard deviation of 0.031522, compared to a wider network like `(64, 32)` which had slightly worse values for accurary at 0.96731 but a lower standard deviation at 0.02108. While the deeper network has a better accuracy, it's only better than the wider network by approximately 0.00731 while having a lower standard deviation, so we can conclude that the wider network is better for this dataset. 

### Max Iterations

In [22]:
# max iter 200
neural = MLPClassifier(max_iter=200, random_state=42).fit(X_train, y_train)
cv_scores = cross_val_score(neural, X, y, cv=5, scoring='accuracy')
cv_scores, np.mean(cv_scores), np.std(cv_scores)

(array([1.        , 0.96774194, 0.90322581, 0.96666667, 0.93333333]),
 0.9541935483870969,
 0.03307635944899851)

In [23]:
# max iter 500
neural = MLPClassifier(max_iter=500, random_state=42).fit(X_train, y_train)
cv_scores = cross_val_score(neural, X, y, cv=5, scoring='accuracy')
cv_scores, np.mean(cv_scores), np.std(cv_scores)

(array([0.96774194, 0.96774194, 0.96774194, 1.        , 0.93333333]),
 0.9673118279569893,
 0.02108843126388174)

In [24]:
# max iter 800
neural = MLPClassifier(max_iter=800, random_state=42).fit(X_train, y_train)
cv_scores = cross_val_score(neural, X, y, cv=5, scoring='accuracy')
cv_scores, np.mean(cv_scores), np.std(cv_scores)

(array([0.96774194, 0.96774194, 0.96774194, 1.        , 0.93333333]),
 0.9673118279569893,
 0.02108843126388174)

Max Iterations is the number of iterations the the network performs. The default is 200 but the rule of thumb is to increase it until the model converges. Now I have probably hidden the warning messages in the final submission but the default of 200 interations and even 500 iterations both warn us that the model has not yet converged. However, we don't get that warning with 800 iterations so we can assume that the model has converged by then. 800 max iterations also gives us the best performance with an accuracy of 0.96731 and a standard deviation of 0.02108.

### Learning Rate

In [25]:
neural = MLPClassifier(learning_rate='adaptive', random_state=42).fit(X_train, y_train)
cv_scores = cross_val_score(neural, X, y, cv=5, scoring='accuracy')
cv_scores, np.mean(cv_scores), np.std(cv_scores)

(array([1.        , 0.96774194, 0.90322581, 0.96666667, 0.93333333]),
 0.9541935483870969,
 0.03307635944899851)

The learning rate defines how the weights update as the model progresses. The default is 'constant' which is a fixed rate whereas 'adaptive' reduces the learning rate when performance plateaus. We can see that an adaptive learning rate has an accuracy of 0.95419 and a standard deviation of 0.03307, which is actually the exact same as the default setting of 'constant' we can conclude that tinkering with this parameter doesn't yield any changes and can be disgarded. 

### Combination

I combined the "best" parameters to see if they created a better neural network. We used a wider network with a max number of iterations of 800.

In [26]:
neural = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=800, random_state=42).fit(X_train, y_train)
cv_scores = cross_val_score(neural, X, y, cv=5, scoring='accuracy')
cv_scores, np.mean(cv_scores), np.std(cv_scores)

(array([0.96774194, 0.96774194, 0.96774194, 1.        , 0.96666667]),
 0.9739784946236559,
 0.013017415871330361)

### Neural Network Results

| Model  | Average of Scores | Standard Deviation in Scores |
|--------|------------------|-----------------------------|
| Default | 0.95419 | 0.03308 |
| Adjusted | 0.97397 | 0.01302 |

We do actually see some pretty noticable changes after adjusting some parameters. The adjusted network outperforms the default one in both average accuracy but it also has a lower standard deviation. Furthermore, it also outperformed all three of the classification models (log, svc, knn) we looked at earlier. The average accuracy is 0.97397 and the standard deviation is 0.01302, making this the highest performing model in our investigation.